In [ ]:
import os, csv, math, random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:

# Experiment
SEED = 0
OUT_DIR = "PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE"
os.makedirs(OUT_DIR, exist_ok=True)

# Data
BATCH_SIZE = 128
NUM_WORKERS = 4
VAL_FRACTION = 0.1

# Train
EPOCHS = 200
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LR_MILESTONES = [60, 120, 160]
LR_GAMMA = 0.2

# Loss (focal + sbece)
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = None          
SBECE_BINS = 15
SBECE_TEMP = 0.01          
BETA_SBECE = 1.0

# Pruning (gating)
LAMBDA_SPARSE = 1e-4        
SPARSITY_RAMP_EPOCHS = 20   
RECOVERY_EPOCHS = 40
PRUNE_RATIOS = [i/10 for i in range(0,10)] 
ACC_DROP_TOL = 0.01         # allow at most 1% absolute acc drop vs unpruned

# Metrics
ECE_BINS = 15
MEAN_SWEEP_BINS = list(range(5, 31))


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


In [ ]:
def get_cifar10_loaders(batch_size=128, val_fraction=0.1, num_workers=4, seed=0):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf)
    test_set   = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

    n_total = len(train_full)
    n_val = int(val_fraction * n_total)
    n_train = n_total - n_val

    g = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(train_full, [n_train, n_val], generator=g)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_cifar10_loaders(
    batch_size=BATCH_SIZE, val_fraction=VAL_FRACTION, num_workers=NUM_WORKERS, seed=SEED
)
print(len(train_loader), len(val_loader), len(test_loader))


In [ ]:
def get_cifar10_bn_loader(batch_size=128, num_workers=4):
    mean = (0.5071, 0.4867, 0.4408)
    std  = (0.2675, 0.2565, 0.2761)

    bn_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_noaug = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=False, transform=bn_tf
    )

    bn_loader = DataLoader(
        train_noaug,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    return bn_loader

In [ ]:
bn_loader = get_cifar10_bn_loader(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)


In [ ]:
class ChannelGate(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Parameter(torch.ones(channels))

    def forward(self, x):
        return x * self.gate.view(1, -1, 1, 1)


class WideBasic(nn.Module):
    def __init__(self, in_planes, planes, dropout_rate, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, padding=1, bias=False)
        self.gate1 = ChannelGate(planes)

        self.dropout = nn.Dropout(p=dropout_rate) if dropout_rate > 0 else nn.Identity()

        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.gate2 = ChannelGate(planes)

        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.gate1(out)
        out = self.dropout(out)

        out = self.conv2(F.relu(self.bn2(out)))
        out = self.gate2(out)

        out += self.shortcut(x)
        return out


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=10, dropout_rate=0.0, num_classes=100):
        super().__init__()
        assert (depth - 4) % 6 == 0, "WRN depth must be 6n+4"
        n = (depth - 4) // 6
        k = widen_factor

        stages = [16, 16*k, 32*k, 64*k]

        self.conv1 = nn.Conv2d(3, stages[0], 3, padding=1, bias=False)
        self.layer1 = self._make_layer(stages[0], stages[1], n, dropout_rate, stride=1)
        self.layer2 = self._make_layer(stages[1], stages[2], n, dropout_rate, stride=2)
        self.layer3 = self._make_layer(stages[2], stages[3], n, dropout_rate, stride=2)
        self.bn1 = nn.BatchNorm2d(stages[3])
        self.fc = nn.Linear(stages[3], num_classes)

        self._init_weights()

    def _make_layer(self, in_planes, planes, num_blocks, dropout_rate, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        curr_in = in_planes
        for s in strides:
            layers.append(WideBasic(curr_in, planes, dropout_rate, stride=s))
            curr_in = planes
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1).view(out.size(0), -1)
        return self.fc(out)


def build_wrn28_10(num_classes=10, dropout=0.0):
    return WideResNet(depth=28, widen_factor=10, dropout_rate=dropout, num_classes=num_classes)


# sanity check
m = build_wrn28_10().to(DEVICE)
print(m(torch.randn(2, 3, 32, 32).to(DEVICE)).shape)  # [2, 100]


## Gating module (for pruning) + “convert WRN to gated WRN”

### GatedConv2d

In [ ]:
# =========================
# Channel-wise gating (BN-safe)
# =========================

class ChannelGate(nn.Module):
    """
    Channel-wise gate applied AFTER BN + ReLU.
    Safe for hardening + BN recalibration.
    """
    def __init__(self, channels):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(channels))

    def gate_values(self):
        return torch.sigmoid(self.logits)

    def forward(self, x):
        g = self.gate_values().view(1, -1, 1, 1)
        return x * g


# =========================
# Gate utilities
# =========================

@torch.no_grad()
def gather_all_gates(model: nn.Module) -> torch.Tensor:
    gates = []
    for m in model.modules():
        if isinstance(m, ChannelGate):
            gates.append(m.gate_values().detach().flatten())
    return torch.cat(gates) if len(gates) > 0 else torch.tensor([])


def gate_l1_sparsity_loss(model: nn.Module):
    """
    L1-style sparsity loss on gates.
    Use with: loss += LAMBDA_SPARSE * gate_l1_sparsity_loss(model)
    """
    vals = []
    for m in model.modules():
        if isinstance(m, ChannelGate):
            vals.append(m.gate_values())
    if len(vals) == 0:
        return torch.tensor(0.0, device=next(model.parameters()).device)
    return torch.cat(vals).mean()


# =========================
# Gate hardening (pruning)
# =========================

@torch.no_grad()
def harden_gates(model, prune_ratio):
    """
    prune_ratio: 0.1 -> prune 10%, keep 90%
    """
    for m in model.modules():
        if isinstance(m, ChannelGate):
            g = m.gate_values()
            k = int((1 - prune_ratio) * g.numel())

            if k <= 0:
                m.logits.fill_(-10.0)
                continue

            thresh = torch.topk(g, k, largest=True).values.min()
            m.logits.copy_(
                torch.where(
                    g >= thresh,
                    torch.tensor(10.0, device=g.device),
                    torch.tensor(-10.0, device=g.device),
                )
            )


## Losses: Focal + SB-ECE

### Focal loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)  # pt = softmax prob of true class
        loss = (1 - pt) ** self.gamma * ce

        if self.alpha is not None:
            # alpha can be scalar or per-class tensor
            if isinstance(self.alpha, (float, int)):
                loss = self.alpha * loss
            else:
                a = self.alpha.to(logits.device)
                loss = a[targets] * loss

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


### SB-ECE loss (binned, L2, soft memberships)

In [ ]:
class SoftBinnedECELoss(nn.Module):
    """
    Soft-binning ECE (binned, L2).
    - anchors at (1/(2m), 3/(2m), ..., (2m-1)/(2m))
    - membership via softmax over - (p - anchor)^2 / temp
    - bin-weight = normalized sum of memberships
    - output = sqrt( sum_b w_b * (conf_b - acc_b)^2 )
    """
    def __init__(self, bins=15, temperature=0.01, eps=1e-5):
        super().__init__()
        self.bins = bins
        self.temperature = temperature
        self.eps = eps
        anchors = np.arange(1.0/(2*bins), 1.0, 1.0/bins)
        self.register_buffer("anchors", torch.tensor(anchors, dtype=torch.float32))

    def forward(self, logits, labels):
        probs = F.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)
        acc = pred.eq(labels).float()

        # [N, m]
        conf_tile = conf.unsqueeze(1)                         # [N,1]
        anchors = self.anchors.unsqueeze(0)                   # [1,m]
        scores = -((conf_tile - anchors) ** 2) / self.temperature
        c = F.softmax(scores, dim=1)                          # memberships

        # sum coeffs per bin: [m]
        sum_c = c.sum(dim=0)                                  # [m]
        sum_c_safe = torch.clamp(sum_c, min=self.eps)

        # soft bin confidence / accuracy: [m]
        conf_b = (c * conf_tile).sum(dim=0) / sum_c_safe
        acc_b  = (c * acc.unsqueeze(1)).sum(dim=0) / sum_c_safe

        # weights normalize to sum 1
        w = sum_c / torch.clamp(sum_c.sum(), min=self.eps)

        # L2 gap (paper uses L2 form)
        sb_ece = torch.sqrt(torch.sum(w * (conf_b - acc_b) ** 2) + self.eps)
        return sb_ece


## Metrics (Acc, NLL, ECE, TS-ECE, KS, mean-sweep CE)

In [ ]:
@torch.no_grad()
def collect_logits_and_labels(model, loader, device):
    model.eval()
    logits_list, labels_list = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        logits_list.append(logits.detach().cpu())
        labels_list.append(y.detach().cpu())
    return torch.cat(logits_list, 0), torch.cat(labels_list, 0)

@torch.no_grad()
def acc_from_logits(logits, labels):
    return float((logits.argmax(1) == labels).float().mean().item())

@torch.no_grad()
def nll_from_logits(logits, labels):
    return float(F.cross_entropy(logits, labels, reduction="mean").item())

@torch.no_grad()
def ece_from_logits(logits, labels, n_bins=15, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    conf, pred = probs.max(1)
    acc = pred.eq(labels).float()

    edges = torch.linspace(0, 1, n_bins + 1, device=logits.device)
    ece = torch.zeros(1, device=logits.device)

    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        inb = (conf > lo) & (conf <= hi)
        p = inb.float().mean()
        if p.item() > 0:
            ece += p * torch.abs(conf[inb].mean() - acc[inb].mean())
    return float(ece.item())

@torch.no_grad()
def ks_calibration_error(logits, labels, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    conf, pred = probs.max(1)
    correct = pred.eq(labels).float()

    conf_sorted, idx = torch.sort(conf)
    corr_sorted = correct[idx]
    denom = torch.arange(1, len(conf_sorted)+1, device=logits.device).float()
    cum_acc = torch.cumsum(corr_sorted, 0) / denom
    cum_conf = torch.cumsum(conf_sorted, 0) / denom
    return float(torch.max(torch.abs(cum_acc - cum_conf)).item())

@torch.no_grad()
def mean_sweep_ce(logits, labels, bin_list, temperature=1.0):
    vals = [ece_from_logits(logits, labels, n_bins=b, temperature=temperature) for b in bin_list]
    return float(np.mean(vals))

def fit_temperature(logits_val_cpu, labels_val_cpu, device, max_iter=50):
    logits = logits_val_cpu.to(device)
    labels = labels_val_cpu.to(device)

    logT = torch.zeros(1, device=device, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.5, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        T = torch.exp(logT)
        loss = F.cross_entropy(logits / T, labels)
        loss.backward()
        return loss

    opt.step(closure)
    T = float(torch.exp(logT).detach().cpu().item())
    return max(1e-3, min(T, 100.0))

@torch.no_grad()
def evaluate_all_metrics(model, val_loader, test_loader, device):
    logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
    logits_test, labels_test = collect_logits_and_labels(model, test_loader, device)

    logits_test = logits_test.to(device)
    labels_test = labels_test.to(device)

    # base
    acc = acc_from_logits(logits_test, labels_test)
    nll = nll_from_logits(logits_test, labels_test)
    ece = ece_from_logits(logits_test, labels_test, n_bins=ECE_BINS)
    ks  = ks_calibration_error(logits_test, labels_test)
    ms  = mean_sweep_ce(logits_test, labels_test, MEAN_SWEEP_BINS)

    # TS
    T = fit_temperature(logits_val, labels_val, device=device)
    ts_nll = nll_from_logits(logits_test / T, labels_test)
    ts_ece = ece_from_logits(logits_test, labels_test, n_bins=ECE_BINS, temperature=T)
    ts_ks  = ks_calibration_error(logits_test, labels_test, temperature=T)
    ts_ms  = mean_sweep_ce(logits_test, labels_test, MEAN_SWEEP_BINS, temperature=T)

    return {
        "acc": acc, "nll": nll, "ece": ece, "ts_ece": ts_ece,
        "ks": ks, "ts_ks": ts_ks,
        "mean_sweep_ce": ms, "ts_mean_sweep_ce": ts_ms,
        "ts_temperature": T,
        "ts_nll": ts_nll,
    }


## Train one epoch with (Focal + β·SB-ECE + λ·sparsity)

In [ ]:
def train_one_epoch_focal_sbece(model, loader, optimizer, device,
                               focal_loss_fn, sbece_loss_fn,
                               beta_sbece=1.0, lambda_sparse=0.0):
    model.train()
    total = 0
    sum_total, sum_focal, sum_sbece, sum_sparse = 0.0, 0.0, 0.0, 0.0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)

        focal = focal_loss_fn(logits, y)
        sbece = sbece_loss_fn(logits, y)
        sparse = gate_l1_sparsity_loss(model) if lambda_sparse > 0 else torch.tensor(0.0, device=device)

        loss = focal + beta_sbece * sbece + lambda_sparse * sparse
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total += bs
        sum_total += float(loss.item()) * bs
        sum_focal += float(focal.item()) * bs
        sum_sbece += float(sbece.item()) * bs
        sum_sparse += float(sparse.item()) * bs

    return {
        "loss": sum_total / total,
        "focal": sum_focal / total,
        "sbece": sum_sbece / total,
        "sparse": sum_sparse / total,
    }


## Calibration-driven pruning: harden gates per layer + BN refresh + recovery

###  Hardening (per-layer top-k keep)

In [ ]:
@torch.no_grad()
def harden_gates_per_layer(model: nn.Module,
                           keep_ratio: float,
                           prune_logit: float = -10.0):
    """
    Per-layer hardening:
    keep_ratio = fraction of channels kept PER gate layer
    Example: keep_ratio=0.9 keeps top 90% channels in each gate
    """
    eps = 1e-6

    for m in model.modules():
        if isinstance(m, ChannelGate):
            g = m.gate_values()          # [C]
            C = g.numel()

            k = max(1, int(round(keep_ratio * C)))
            thresh = torch.topk(g, k, largest=True).values.min()
            keep = g >= thresh

            # convert soft gates → hard logits
            g_clamped = g.clamp(eps, 1 - eps)
            logits_keep = torch.log(g_clamped / (1 - g_clamped))

            m.logits.data[keep] = logits_keep[keep]
            m.logits.data[~keep] = prune_logit



### BN refresh

In [ ]:
@torch.no_grad()
def reset_bn_running_stats(model: nn.Module):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.running_mean.zero_()
            m.running_var.fill_(1)

@torch.no_grad()
def update_bn_stats(model: nn.Module, loader, device, num_batches=100):
    model.train()
    n = 0
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        _ = model(x)
        n += 1
        if n >= num_batches:
            break

### Recovery fine-tune (same loss but sparsity off)

In [ ]:
def recovery_finetune(model, train_loader, device, lr, weight_decay,
                      focal_loss_fn, sbece_loss_fn, beta_sbece, epochs=40):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM,
                                weight_decay=weight_decay, nesterov=True)
    for ep in range(1, epochs+1):
        tr = train_one_epoch_focal_sbece(
            model, train_loader, optimizer, device,
            focal_loss_fn, sbece_loss_fn,
            beta_sbece=beta_sbece, lambda_sparse=0.0
        )
        if ep % 5 == 0:
            print(f"   recovery epoch {ep:02d}/{epochs} done | loss={tr['loss']:.4f}")


## FULL RUN COMMAND: train (focal+sbece) + prune sweep + CSV

In [ ]:
def run_focal_sbece_training_and_pruning_one_seed():
    # ---- loaders ----
    train_loader, val_loader, test_loader = get_cifar10_loaders(
        batch_size=BATCH_SIZE,
        val_fraction=VAL_FRACTION,
        num_workers=NUM_WORKERS,
        seed=SEED
    )

    # ---- model ----
    base_model = build_wrn28_10(num_classes=10, dropout=0.0).to(DEVICE)

    # ---- losses ----
    focal_fn = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA).to(DEVICE)
    sbece_fn = SoftBinnedECELoss(
        bins=SBECE_BINS,
        temperature=SBECE_TEMP
    ).to(DEVICE)

    # ---- optimizer / scheduler ----
    optimizer = torch.optim.SGD(
        base_model.parameters(),
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        nesterov=True
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=LR_MILESTONES,
        gamma=LR_GAMMA
    )

    print("\n==============================")
    print("TRAINING UNPRUNED BASELINE")
    print("==============================")

    # ---- TRAIN UNPRUNED ----
    for epoch in range(1, EPOCHS + 1):
        lambda_eff = LAMBDA_SPARSE * min(1.0, epoch / SPARSITY_RAMP_EPOCHS)

        tr = train_one_epoch_focal_sbece(
            base_model,
            train_loader,
            optimizer,
            DEVICE,
            focal_fn,
            sbece_fn,
            beta_sbece=BETA_SBECE,
            lambda_sparse=lambda_eff
        )
        scheduler.step()

        # ---- progress print ----
        if epoch % 2 == 0 or epoch == 1:
            base_model.eval()
            m = evaluate_all_metrics(
                base_model,
                val_loader,
                test_loader,
                DEVICE
            )

            g_all = gather_all_gates(base_model)
            gate_mean = float(g_all.mean().item()) if g_all.numel() else 0.0
            gate_density = float((g_all > 0.5).float().mean().item()) if g_all.numel() else 0.0

            # NEW: sparsity diagnostics
            sparse_val = gate_l1_sparsity_loss(base_model).item()
            lambda_eff = LAMBDA_SPARSE * min(1.0, epoch / SPARSITY_RAMP_EPOCHS)

            print(
                f"[BASE] Epoch {epoch:03d}/{EPOCHS} | "
                f"loss={tr['loss']:.4f} "
                f"(focal={tr['focal']:.4f}, sbece={tr['sbece']:.4f}, sparse={sparse_val:.4f}, λ={lambda_eff:.1e}) | "
                f"acc={m['acc']:.4f} ece={m['ece']:.4f} ks={m['ks']:.4f} | "
                f"gate_mean={gate_mean:.3f} density@0.5={gate_density:.3f}"
            )


    # ---- eval unpruned ----
    base_metrics = evaluate_all_metrics(
        base_model,
        val_loader,
        test_loader,
        DEVICE
    )
    print("\nUNPRUNED METRICS:", base_metrics)

    unpruned_path = os.path.join(
        OUT_DIR,
        "wrn28_10_cifar10_unpruned.pth"
    )
    torch.save(
        {"state_dict": base_model.state_dict(),
         "metrics": base_metrics},
        unpruned_path
    )
    print("Saved unpruned ->", unpruned_path)

    

In [ ]:
csv_path = run_focal_sbece_training_and_pruning_one_seed()
print("CSV saved at:", csv_path)


In [ ]:
# ============================================================
# FULL PIPELINE: WRN-28-10 + ChannelGate + Focal + SB-ECE
# CIFAR-100 | Baseline checkpoint → Gate harden → BN recalib → Recovery (min-epochs + early stop)
# ============================================================

import os, csv, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# expects these to exist in your notebook:
#   build_wrn28_10
#   ChannelGate
#   get_cifar100_loaders
#   get_cifar100_bn_loader
#   train_one_epoch_focal_sbece
#   evaluate_all_metrics
#   reset_bn_running_stats
#   update_bn_stats
#   FocalLoss
#   SoftBinnedECELoss
#
# plus globals you already have:
#   SEED, BATCH_SIZE, VAL_FRACTION, NUM_WORKERS
#   LR, MOMENTUM, WEIGHT_DECAY
#   FOCAL_GAMMA, SBECE_BINS, SBECE_TEMP, BETA_SBECE
#   ACC_DROP_TOL

# ------------------------------------------------------------
# GATE HARDENING + FREEZING
# ------------------------------------------------------------

@torch.no_grad()
def harden_binary_gates(
    model: nn.Module,
    keep_ratio: float,
    on_logit: float = 10.0,
    off_logit: float = -10.0,
):
    """
    Per-layer hardening for ChannelGate.
    Keeps top keep_ratio channels per gate (per gate module).
    """
    for m in model.modules():
        if isinstance(m, ChannelGate):
            g = m.gate_values().detach()
            C = g.numel()
            k = max(1, int(round(keep_ratio * C)))

            idx = torch.topk(g, k, largest=True).indices
            keep = torch.zeros_like(g, dtype=torch.bool)
            keep[idx] = True

            m.logits.data[keep]  = on_logit
            m.logits.data[~keep] = off_logit


def set_gate_trainable(model: nn.Module, trainable: bool):
    for m in model.modules():
        if isinstance(m, ChannelGate):
            m.logits.requires_grad_(trainable)


@torch.no_grad()
def gate_stats(model: nn.Module):
    """Returns (gate_mean, density@0.5) across all ChannelGate modules."""
    gs = []
    for m in model.modules():
        if isinstance(m, ChannelGate):
            gs.append(m.gate_values().detach().flatten())
    if len(gs) == 0:
        return 0.0, 0.0
    g = torch.cat(gs, 0)
    mean = float(g.mean().item())
    dens = float((g > 0.5).float().mean().item())
    return mean, dens


# ------------------------------------------------------------
# CHECKPOINT LOADING
# ------------------------------------------------------------

def load_checkpoint_strict(model: nn.Module, ckpt_path: str, device: str):
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    else:
        sd = ckpt

    missing, unexpected = model.load_state_dict(sd, strict=True)
    print(f"[load] {ckpt_path}")
    print(f"        missing={len(missing)} unexpected={len(unexpected)}")
    if missing:
        print("        missing keys (first 20):", missing[:20])
    if unexpected:
        print("        unexpected keys (first 20):", unexpected[:20])

    model.to(device)
    return model


# ------------------------------------------------------------
# RECOVERY FINETUNING (LONG + MIN-EPOCHS + EARLY STOP)
# ------------------------------------------------------------

def recovery_finetune_long_earlystop(
    model,
    train_loader,
    val_loader,
    test_loader,
    device,
    lr,
    weight_decay,
    focal_loss_fn,
    sbece_loss_fn,
    beta_sbece,
    out_dir,
    tag,
    max_epochs=160,
    eval_every=5,
    patience=10,
    min_delta=1e-4,
    no_stop_before_ep=20,         
    print_every=5,                
):
    os.makedirs(out_dir, exist_ok=True)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=MOMENTUM,
        weight_decay=weight_decay,
        nesterov=True
    )

    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[80, 120], gamma=0.1
    )

    best_val_acc = -1.0
    best_state = None
    bad_steps = 0
    rows = []
    t0 = time.time()

    for ep in range(1, max_epochs + 1):
        tr = train_one_epoch_focal_sbece(
            model,
            train_loader,
            optimizer,
            device,
            focal_loss_fn,
            sbece_loss_fn,
            beta_sbece=beta_sbece,
            lambda_sparse=0.0  # 
        )
        scheduler.step()

        row = {
            "epoch": ep,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "train_loss": float(tr["loss"]),
            "train_focal": float(tr["focal"]),
            "train_sbece": float(tr["sbece"]),
        }

        do_eval = (ep % eval_every == 0) or (ep == 1) or (ep == max_epochs)
        if do_eval:
            model.eval()
            m = evaluate_all_metrics(model, val_loader, test_loader, device)

            row.update({
                "val_acc": float(m["acc"]),
                "val_nll": float(m["nll"]),
                "val_ece": float(m["ece"]),
                "val_ts_ece": float(m["ts_ece"]),
                "val_ks": float(m.get("ks", float("nan"))),
                "val_mean_sweep_ce": float(m.get("mean_sweep_ce", float("nan"))),
            })

            # best tracking
            improved = row["val_acc"] > (best_val_acc + min_delta)
            if improved:
                best_val_acc = row["val_acc"]
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                bad_steps = 0
            else:
                #  only start counting bad steps after warmup
                if ep >= no_stop_before_ep:
                    bad_steps += 1

            # logging (don’t spam: print every print_every epochs)
            if (ep == 1) or (ep % print_every == 0) or (ep == max_epochs) or improved:
                print(
                    f"   [{tag}] ep {ep:03d}/{max_epochs} | "
                    f"train_loss={row['train_loss']:.4f} (focal={row['train_focal']:.4f}, sbece={row['train_sbece']:.4f}) | "
                    f"val_acc={row['val_acc']:.4f} best={best_val_acc:.4f} "
                    f"bad={bad_steps}/{patience} (stop_after_ep>={no_stop_before_ep})"
                )

            if (ep >= no_stop_before_ep) and (bad_steps >= patience):
                print(f"   [{tag}] Early stopping at epoch {ep}")
                break

        rows.append(row)

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    # final metrics on best
    model.eval()
    final_metrics = evaluate_all_metrics(model, val_loader, test_loader, device)

    curves_path = os.path.join(out_dir, f"curves_{tag}.csv")
    pd.DataFrame(rows).to_csv(curves_path, index=False)

    best_path = os.path.join(out_dir, f"best_{tag}.pth")
    torch.save(
        {"state_dict": model.state_dict(), "final_metrics": final_metrics, "tag": tag},
        best_path
    )

    print(f"   [{tag}] Saved curves -> {curves_path}")
    print(f"   [{tag}] Saved best   -> {best_path}")
    print(f"   [{tag}] Time: {(time.time() - t0) / 60:.1f} min")

    return final_metrics, curves_path, best_path


# ------------------------------------------------------------
# PRUNING SWEEP FROM BASELINE CHECKPOINT
# ------------------------------------------------------------

def run_pruning_sweep_from_checkpoint_one_seed(
    ckpt_path: str,
    out_dir: str,
    prune_ratios,
    max_recovery_epochs=40,
    eval_every=1,
    patience=10,
    no_stop_before_ep=10,          
    bn_num_batches=100, 
    print_every=1
):
    os.makedirs(out_dir, exist_ok=True)

    # loaders
    train_loader, val_loader, test_loader = get_cifar10_loaders(
        batch_size=BATCH_SIZE,
        val_fraction=VAL_FRACTION,
        num_workers=NUM_WORKERS,
        seed=SEED
    )


    # load baseline
    base_model = build_wrn28_10(num_classes=10, dropout=0.0).to(DEVICE)
    base_model = load_checkpoint_strict(base_model, ckpt_path, DEVICE)

    base_metrics = evaluate_all_metrics(base_model, val_loader, test_loader, DEVICE)
    gm, gd = gate_stats(base_model)
    print("BASELINE:", base_metrics, f"| gate_mean={gm:.3f} density@0.5={gd:.3f}")

    # keep a CPU copy
    base_state = {k: v.detach().cpu() for k, v in base_model.state_dict().items()}

    # losses for recovery (create once)
    focal_fn = FocalLoss(FOCAL_GAMMA).to(DEVICE)
    sbece_fn = SoftBinnedECELoss(SBECE_BINS, SBECE_TEMP).to(DEVICE)

    # CSV
    csv_path = os.path.join(out_dir, "metrics_pruning_sweep.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "prune_ratio", "keep_ratio",
            "post_harden_acc", "post_harden_ece", "post_harden_ks",
            "final_acc", "final_ece", "final_ts_ece",
            "accepted_acc_drop<=tol",
            "best_ckpt_path", "curves_csv_path"
        ])

        for prune_ratio in prune_ratios:
            keep_ratio = 1.0 - prune_ratio
            tag = f"prune{int(prune_ratio*100):02d}_keep{int(keep_ratio*100):02d}"

            print("\n" + "=" * 70)
            print(f"[{tag}] Hardening + BN recalibration + Recovery")
            print("=" * 70)

            # fresh model
            model_p = build_wrn28_10(num_classes=10, dropout=0.0).to(DEVICE)
            model_p.load_state_dict(base_state, strict=True)

            # harden + freeze
            harden_binary_gates(model_p, keep_ratio=keep_ratio)
            set_gate_trainable(model_p, False)

            # BN recalibration (important after hardening)
            reset_bn_running_stats(model_p)
            update_bn_stats(model_p, bn_loader, DEVICE, num_batches=bn_num_batches)

            # post-harden diagnostic (on val/test loaders)
            model_p.eval()
            mh = evaluate_all_metrics(model_p, val_loader, test_loader, DEVICE)
            print(f"-> Post-harden eval: acc={mh['acc']:.4f} nll={mh['nll']:.4f} ece={mh['ece']:.4f}")

            # recovery
            m, curves_csv, best_ckpt = recovery_finetune_long_earlystop(
                model_p,
                train_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                device=DEVICE,
                lr=LR * 0.1,
                weight_decay=WEIGHT_DECAY,
                focal_loss_fn=focal_fn,
                sbece_loss_fn=sbece_fn,
                beta_sbece=BETA_SBECE,
                out_dir=out_dir,
                tag=tag,
                max_epochs=max_recovery_epochs,
                eval_every=eval_every,
                patience=patience,
                no_stop_before_ep=no_stop_before_ep,
                print_every=5,
            )

            accepted = (base_metrics["acc"] - m["acc"]) <= ACC_DROP_TOL
            print(f"-> Final: acc={m['acc']:.4f} ece={m['ece']:.4f} ts_ece={m['ts_ece']:.4f} | accepted={accepted}")

            writer.writerow([
                prune_ratio, keep_ratio,
                mh["acc"], mh["ece"], mh.get("ks", float("nan")),
                m["acc"], m["ece"], m["ts_ece"],
                accepted,
                best_ckpt, curves_csv
            ])

    print("DONE ->", csv_path)
    return csv_path


In [ ]:
csv_path = run_pruning_sweep_from_checkpoint_one_seed(
    ckpt_path="PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/wrn28_10_cifar10_unpruned.pth",
    out_dir="PRUNING/WRN28-10_CIFAR100/FOCAL_SBECE_PRUNE/SOFT_GATE/SWEEP",
    prune_ratios=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8,0.9],
)
print("CSV saved at:", csv_path)



In [ ]:
@torch.no_grad()
def evaluate_checkpoint(
    ckpt_path: str,
    val_loader,
    test_loader,
    device,
):
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt

    model = build_wrn28_10(num_classes=10, dropout=0.0).to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    metrics = evaluate_all_metrics(model, val_loader, test_loader, device)
    return metrics


In [ ]:
import os
import pandas as pd

def evaluate_baseline_and_pruned_models(
    baseline_ckpt: str,
    sweep_dir: str,
    prune_ratios,
    device,
):
    _, val_loader, test_loader = get_cifar10_loaders(
        batch_size=BATCH_SIZE,
        val_fraction=VAL_FRACTION,
        num_workers=NUM_WORKERS,
        seed=SEED,
    )

    rows = []

    # ------------------ BASELINE ------------------
    base_metrics = evaluate_checkpoint(
        baseline_ckpt, val_loader, test_loader, device
    )

    rows.append({
        "model": "baseline",
        "prune_ratio": 0.0,
        **base_metrics
    })

    for p in prune_ratios:


        tag = f"prune{int(p*100):02d}_keep{int((1-p)*100):02d}"
        ckpt_path = os.path.join(sweep_dir, f"best_{tag}.pth")

        if not os.path.exists(ckpt_path):
            print(f"[WARN] Missing: {ckpt_path}")
            continue

        m = evaluate_checkpoint(
            ckpt_path, val_loader, test_loader, device
        )

        rows.append({
            "model": tag,
            "prune_ratio": p,
            "keep_ratio": 1.0 - p,
            **m
        })

    return pd.DataFrame(rows)


In [ ]:
BASELINE_CKPT = (
    "PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/"
    "wrn28_10_cifar10_unpruned.pth"
)

SWEEP_DIR = (
    "PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/SWEEP"
)

df = evaluate_baseline_and_pruned_models(
    baseline_ckpt=BASELINE_CKPT,
    sweep_dir=SWEEP_DIR,
    prune_ratios=[0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7, 0.8, 0.9],
    device=DEVICE,
)

df


## Bootsrapping 

In [ ]:
# --------------------------------------------------
# Bootstrap utility (FULL CIFAR metric set)
# --------------------------------------------------

def bootstrap_metrics_from_logits(
    logits_val_cpu,
    labels_val_cpu,
    logits_test_cpu,
    labels_test_cpu,
    device,
    n_boot=300,
):
    """
    Bootstrap uncertainty for:
    acc, nll, ece, ks, mean_sweep_ce,
    ts_ece, ts_nll, ts_ks, ts_mean_sweep_ce
    """

    N = logits_test_cpu.shape[0]

    results = []

    for _ in range(n_boot):

        # ---- Resample TEST set ----
        idx = torch.randint(0, N, (N,))
        logits_test_b = logits_test_cpu[idx].to(device)
        labels_test_b = labels_test_cpu[idx].to(device)

        # ---- Base metrics ----
        acc = acc_from_logits(logits_test_b, labels_test_b)
        nll = nll_from_logits(logits_test_b, labels_test_b)
        ece = ece_from_logits(logits_test_b, labels_test_b, n_bins=ECE_BINS)
        ks  = ks_calibration_error(logits_test_b, labels_test_b)
        ms  = mean_sweep_ce(logits_test_b, labels_test_b, MEAN_SWEEP_BINS)

        # ---- Temperature Scaling (fit on VAL each time) ----
        T = fit_temperature(
            logits_val_cpu,
            labels_val_cpu,
            device=device
        )

        ts_nll = nll_from_logits(logits_test_b / T, labels_test_b)
        ts_ece = ece_from_logits(logits_test_b, labels_test_b, n_bins=ECE_BINS, temperature=T)
        ts_ks  = ks_calibration_error(logits_test_b, labels_test_b, temperature=T)
        ts_ms  = mean_sweep_ce(logits_test_b, labels_test_b, MEAN_SWEEP_BINS, temperature=T)

        results.append({
            "acc": acc,
            "nll": nll,
            "ece": ece,
            "ks": ks,
            "mean_sweep_ce": ms,
            "ts_ece": ts_ece,
            "ts_nll": ts_nll,
            "ts_ks": ts_ks,
            "ts_mean_sweep_ce": ts_ms,
        })

    df_boot = pd.DataFrame(results)

    std_dict = {k + "_std": float(df_boot[k].std()) for k in df_boot.columns}

    return std_dict


In [ ]:
@torch.no_grad()
def evaluate_checkpoint_with_bootstrap(
    ckpt_path: str,
    val_loader,
    test_loader,
    device,
    n_boot=300,
):
    # -----------------------
    # Load model
    # -----------------------
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt

    model = build_wrn28_10(num_classes=10, dropout=0.0).to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    # -----------------------
    # Collect logits once
    # -----------------------
    logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
    logits_test, labels_test = collect_logits_and_labels(model, test_loader, device)

    logits_val_cpu = logits_val.detach().cpu()
    labels_val_cpu = labels_val.detach().cpu()
    logits_test_cpu = logits_test.detach().cpu()
    labels_test_cpu = labels_test.detach().cpu()

    logits_test = logits_test.to(device)
    labels_test = labels_test.to(device)

    # -----------------------
    # Compute mean metrics
    # -----------------------
    metrics = evaluate_all_metrics(model, val_loader, test_loader, device)

    # -----------------------
    # Bootstrap std
    # -----------------------
    boot = bootstrap_metrics_from_logits(
        logits_val_cpu,
        labels_val_cpu,
        logits_test_cpu,
        labels_test_cpu,
        device,
        n_boot=n_boot
    )

    # Merge mean + std
    return {**metrics, **boot}


In [ ]:
def evaluate_baseline_and_pruned_models(
    baseline_ckpt: str,
    sweep_dir: str,
    prune_ratios,
    device,
    n_boot=300,
):
    _, val_loader, test_loader = get_cifar10_loaders(
        batch_size=BATCH_SIZE,
        val_fraction=VAL_FRACTION,
        num_workers=NUM_WORKERS,
        seed=SEED,
    )

    rows = []

    # ------------------ BASELINE ------------------
    print("\nEvaluating BASELINE")
    base_metrics = evaluate_checkpoint_with_bootstrap(
        baseline_ckpt,
        val_loader,
        test_loader,
        device,
        n_boot=n_boot
    )

    rows.append({
        "model": "baseline",
        "prune_ratio": 0.0,
        "keep_ratio": 1.0,
        **base_metrics
    })

    # ------------------ PRUNED MODELS ------------------
    for p in prune_ratios:

        tag = f"prune{int(p*100):02d}_keep{int((1-p)*100):02d}"
        ckpt_path = os.path.join(sweep_dir, f"best_{tag}.pth")

        if not os.path.exists(ckpt_path):
            print(f"[WARN] Missing: {ckpt_path}")
            continue

        print(f"\nEvaluating {tag}")

        m = evaluate_checkpoint_with_bootstrap(
            ckpt_path,
            val_loader,
            test_loader,
            device,
            n_boot=n_boot
        )

        rows.append({
            "model": tag,
            "prune_ratio": p,
            "keep_ratio": 1.0 - p,
            **m
        })

    df = pd.DataFrame(rows)

    return df


In [ ]:
BASELINE_CKPT = (
    "PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/"
    "wrn28_10_cifar10_unpruned.pth"
)

SWEEP_DIR = (
    "PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/SWEEP"
)

df = evaluate_baseline_and_pruned_models(
    baseline_ckpt=BASELINE_CKPT,
    sweep_dir=SWEEP_DIR,
    prune_ratios=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
    device=DEVICE,
    n_boot=300   
)

df



In [ ]:
# --------------------------------------------------
# RAW METRICS TABLE (mean ± std)
# --------------------------------------------------

df_raw = df.copy()

# Accuracy (%)
df_raw["acc (%)"] = (
    (df_raw["acc"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["acc_std"] * 100).round(2).astype(str)
)

# NLL (raw scale)
df_raw["nll"] = (
    df_raw["nll"].round(4).astype(str)
    + " ± "
    + df_raw["nll_std"].round(4).astype(str)
)

# ECE (%)
df_raw["ece (%)"] = (
    (df_raw["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ece_std"] * 100).round(2).astype(str)
)

# KS (%)
df_raw["ks (%)"] = (
    (df_raw["ks"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ks_std"] * 100).round(2).astype(str)
)

# Mean Sweep CE (%)
df_raw["mean_sweep_ce (%)"] = (
    (df_raw["mean_sweep_ce"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["mean_sweep_ce_std"] * 100).round(2).astype(str)
)

df_raw = df_raw[[
    "prune_ratio",
    "acc (%)",
    "nll",
    "ece (%)",
    "ks (%)",
    "mean_sweep_ce (%)"
]]

df_raw.to_csv("PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/SWEEP/cifar10_raw_metrics_paper.csv", index=False)

print("Saved -> cifar_raw_metrics_paper.csv")
df_raw


In [ ]:
# --------------------------------------------------
# TEMPERATURE SCALED METRICS TABLE (mean ± std)
# --------------------------------------------------

df_ts = df.copy()

# Raw ECE (%)
df_ts["raw_ece (%)"] = (
    (df_ts["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ece_std"] * 100).round(2).astype(str)
)

# Raw NLL
df_ts["raw_nll"] = (
    df_ts["nll"].round(4).astype(str)
    + " ± "
    + df_ts["nll_std"].round(4).astype(str)
)

# TS-ECE (%)
df_ts["ts_ece (%)"] = (
    (df_ts["ts_ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ts_ece_std"] * 100).round(2).astype(str)
)

# TS-NLL
df_ts["ts_nll"] = (
    df_ts["ts_nll"].round(4).astype(str)
    + " ± "
    + (df_ts["ts_nll_std"] * 100).round(4).astype(str)
)

# Temperature (no std typically reported)
df_ts["temperature"] = df_ts["ts_temperature"].round(3)

df_ts = df_ts[[
    "prune_ratio",
    "raw_ece (%)",
    "raw_nll",
    "ts_ece (%)",
    "ts_nll",
    "temperature"
]]

df_ts.to_csv("PRUNING/WRN28-10_CIFAR10/FOCAL_SBECE_PRUNE/SOFT_GATE/SWEEP/cifar10_temperature_scaled_metrics_paper.csv", index=False)

print("Saved -> cifar_temperature_scaled_metrics_paper.csv")
df_ts
